# Visualization: Matplotlib and Seaborn

**M0 · Toolbox & Submission Craft** — [AI Olympiad Curriculum](https://abzlitochka.github.io/ai-olympiad-curriculum)

~35 min

## Matplotlib: figures and axes

Matplotlib is the base plotting library. Use the object-oriented style: create a **figure** with one or more **axes**, and draw on each axes — here *Axes* means one subplot panel, unrelated to NumPy's 'axis' from Lesson 5.



In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set(xlabel="epoch", ylabel="loss", title="Learning curves")
axes[0].legend()

axes[1].hist(df["age"], bins=30)
axes[1].set_title("Age distribution")
plt.tight_layout(); plt.show()

The main plot types: `plot` (lines, learning curves), `scatter` (two variables, embedding vectors — defined properly in M2), `hist` (distribution of one variable), `bar` (counts per category), `imshow` (images, matrices, spectrograms).



## Seaborn: statistical plots from DataFrames

Seaborn is built on Matplotlib and works directly with pandas columns. One line gives plots that would take ten in Matplotlib.



In [ ]:
import seaborn as sns
sns.histplot(data=df, x="income", hue="target", kde=True)   # distribution per class
sns.boxplot(data=df, x="city", y="income")                  # spread + outliers per group
sns.scatterplot(data=df, x="age", y="income", hue="target")
sns.countplot(data=df, x="target")                          # class balance
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm")   # corr: how two columns move together (−1 opposite, +1 same direction)
sns.pairplot(df[["age", "income", "target"]], hue="target")

## Plots every ML project needs

- **Target distribution** (countplot / histplot): is the data imbalanced or skewed?
- **Feature vs target** (boxplot, histplot with hue): which features separate the classes?
- **Correlation heatmap**: redundant features and possible **leakage** — a feature that secretly contains the answer, e.g. an id that encodes the label.
- **Learning curves** (train vs validation loss): over- or underfitting?
- **Confusion matrix** (heatmap): which classes get confused?
- **2-D embedding scatter** (PCA / t-SNE / UMAP coloured by label): do classes form clusters?



In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_val, y_pred)
sns.heatmap(cm, annot=True, fmt="d", xticklabels=classes, yticklabels=classes)
plt.xlabel("predicted"); plt.ylabel("true")

> **Note.** Look at the data before modelling. A few plots often reveal leakage, outliers or label errors that no model will fix.



## Readable figures and saving

A figure bound for a report needs axis labels, a title, a legend, and enough resolution. `figsize` is in inches, `dpi` is dots per inch — `figsize=(8, 4), dpi=150` is a good default. `plt.tight_layout()` stops labels from overlapping.



In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), dpi=150)
ax.plot(history["val_loss"], marker="o", label="val")
ax.set(xlabel="epoch", ylabel="loss", title="Validation loss")
ax.grid(alpha=0.3); ax.legend()
ax.set_yscale("log")             # for losses spanning orders of magnitude
fig.savefig("curve.png", bbox_inches="tight")   # save BEFORE plt.show()

- `plt.show()` clears the current figure — `savefig` after `show()` writes an empty PNG.
- Log scale hides nothing honestly: if a curve only looks good in log, say so and show the raw curve too.
- One message per figure. Two unrelated y-axes on one plot (`twinx`) confuse more than they reveal.



### ✏️ You want to see whether `income` is distributed differently for each class of `target`. Best plot?

**A.** sns.lineplot(data=df, x='income', y='target')
**B.** sns.histplot(data=df, x='income', hue='target')
**C.** sns.heatmap(df[['income', 'target']].corr())
**D.** plt.hist(df['income'], bins=30)

<details><summary>Answer</summary>

**B.** sns.histplot(data=df, x='income', hue='target')

A histogram per class (hue) compares the distributions directly.

</details>

### ✏️ Training loss keeps falling while validation loss rises after epoch 10. Which plot shows this?

**A.** Residual plot of predicted vs true values
**B.** Learning curves: train and validation loss vs epoch
**C.** Confusion matrix on the validation set at epoch 10
**D.** Histogram of per-sample validation losses

<details><summary>Answer</summary>

**B.** Learning curves: train and validation loss vs epoch

Learning curves are the standard way to spot overfitting.

</details>

### ✏️ In `fig, axes = plt.subplots(2, 3)`, what is `axes`?

**A.** A single Axes that holds all six panels
**B.** A 2×3 array of Axes objects, one per subplot
**C.** The Figure object containing six panels
**D.** A list of 6 separate Figure objects

<details><summary>Answer</summary>

**B.** A 2×3 array of Axes objects, one per subplot

subplots returns the figure and a grid of axes; you draw on each one, e.g. axes[0, 1].plot(...).

</details>

---

## ✅ Self-check tasks

Implement the function in the 'your code' cell, then run the check cell — it runs randomized tests and shows the first mismatch. Run the helper cell below once first.

In [ ]:
# ── task-checking helper cell — run once ──
import copy
import numpy as np
import math, statistics, itertools, collections, functools, heapq, random

try:
    import sys; sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

_L = {"noFn":"❓ Define a function `{e}` in the solution cell and run it","runErr":"❌ Test {i}: {t}: {m}","wrong":"❌ Test {i}: {r}","badRef":"⚠️ Check harness broke (generator/reference): {t}: {m}","inL":"Input","expL":"Expected","gotL":"Got","ok":"✅ Correct! All {n} tests passed."}

def _nb_copy_kw(kw):
    return {k: (v.copy() if isinstance(v, np.ndarray) else copy.deepcopy(v)) for k, v in kw.items()}

def _nb_jsafe(x, depth=0):
    if depth > 6: return "…"
    if isinstance(x, np.ndarray): return _nb_jsafe(x.tolist(), depth + 1)
    if isinstance(x, np.integer): return int(x)
    if isinstance(x, np.floating): return float(x)
    if isinstance(x, np.bool_): return bool(x)
    if isinstance(x, float): return round(x, 6) if math.isfinite(x) else str(x)
    if isinstance(x, (list, tuple)): return [_nb_jsafe(v, depth + 1) for v in x]
    if isinstance(x, dict): return {str(k): _nb_jsafe(v, depth + 1) for k, v in x.items()}
    if isinstance(x, (int, str, bool)) or x is None: return x
    return repr(x)

def _nb_cmp(exp, got, path=""):
    if isinstance(exp, np.ndarray) or isinstance(got, np.ndarray):
        e, g = np.asarray(exp), np.asarray(got)
        if e.shape != g.shape:
            return f"shape {tuple(g.shape)} != expected {tuple(e.shape)}"
        try:
            if np.issubdtype(e.dtype, np.number) or np.issubdtype(g.dtype, np.number):
                ef, gf = e.astype(float), g.astype(float)
                close = np.isclose(gf, ef, atol=1e-6, rtol=1e-5, equal_nan=True)
                if not bool(close.all()):
                    i = tuple(int(v) for v in np.argwhere(~close)[0])
                    return f"{path}first mismatch at index {i}: got {gf[i]:.6g} vs expected {ef[i]:.6g}"
            elif not np.array_equal(g, e):
                return f"{path}values differ"
        except Exception as ex:
            return f"{path}cannot compare: {ex}"
        return None
    if isinstance(exp, (int, float)) and not isinstance(exp, bool) and isinstance(got, (int, float)) and not isinstance(got, bool):
        if not np.isclose(float(exp), float(got), atol=1e-6, rtol=1e-5):
            return f"{path}{got} != expected {exp}"
        return None
    if isinstance(exp, (list, tuple)) and isinstance(got, (list, tuple)):
        if len(exp) != len(got):
            return f"{path}length {len(got)} != expected {len(exp)}"
        for i, (a, b) in enumerate(zip(exp, got)):
            r = _nb_cmp(a, b, path + f"[{i}].")
            if r: return r
        return None
    if isinstance(exp, dict) and isinstance(got, dict):
        if set(exp) != set(got):
            return f"{path}keys differ"
        for k in exp:
            r = _nb_cmp(exp[k], got[k], path + f"[{k!r}].")
            if r: return r
        return None
    if exp != got:
        return f"{path}{got!r} != expected {exp!r}"
    return None

def _check_task(g, ref_src, entry, n=20, seed=0):
    """g — globals() ноутбука; ref_src — gen() + эталонное решение."""
    fn = g.get(entry)
    if not callable(fn):
        print(_L["noFn"].format(e=entry)); return
    ns = {"np": np, "math": math, "statistics": statistics, "itertools": itertools,
          "collections": collections, "functools": functools, "heapq": heapq, "random": random}
    try:
        exec(ref_src, ns)
        gen, ref = ns["gen"], ns[entry]
    except Exception as e:
        print(_L["badRef"].format(t=type(e).__name__, m=e)); return
    rng = np.random.default_rng(seed)
    for i in range(n):
        try:
            kw = gen(rng)
        except Exception as e:
            print(_L["badRef"].format(t=type(e).__name__, m=e)); return
        try:
            got = fn(**_nb_copy_kw(kw))
        except Exception as e:
            print(_L["runErr"].format(i=i + 1, t=type(e).__name__, m=e)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))); return
        exp = ref(**_nb_copy_kw(kw))
        r = _nb_cmp(exp, got)
        if r:
            print(_L["wrong"].format(i=i + 1, r=r)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))
                  + "\n" + _L["expL"] + ": " + repr(_nb_jsafe(exp))
                  + "\n" + _L["gotL"] + ": " + repr(_nb_jsafe(got))); return
    print(_L["ok"].format(n=n))

### Task 1 — Confusion matrix from labels

●○○ · ~8 min

> A confusion matrix C counts, for every pair (true class i, predicted class j), how many samples landed there: row i is the distribution of what truly-class-i items were called. The diagonal holds the correct predictions, and the largest off-diagonal cell tells you exactly which two classes the model mixes up — that is where you go hunting for better features. `np.add.at(C, (y_true, y_pred), 1)` builds it in a single vectorized pass.

Implement `confusion_matrix(y_true, y_pred, k)` → a (k, k) integer array where `C[i, j]` is the number of samples with true label `i` and predicted label `j`.

```
y_true=[0,0,1], y_pred=[0,1,1], k=2 → [[1,1],[0,1]]
```

<details><summary>Hint</summary>

C = np.zeros((k, k), int); np.add.at(C, (y_true, y_pred), 1) — np.add.at handles repeated index pairs correctly, unlike C[a, b] += 1.

</details>


In [ ]:
import numpy as np

def confusion_matrix(y_true, y_pred, k):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    C = np.zeros((k, k), dtype=int)
    # your code
    return C

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    n, k = int(rng.integers(10, 60)), int(rng.integers(2, 7))\n    # correlated preds — pairs (i,j) repeat heavily, so C[a,b] += 1\n    # without np.add.at (which drops duplicate counts) must fail\n    hi = k - 1 if rng.random() < 0.35 else k   # sometimes class k-1 absent -> k must come from the arg\n    y_true = rng.integers(0, hi, n)\n    same = rng.random(n) < 0.7\n    y_pred = np.where(same, y_true, rng.integers(0, hi, n))\n    return {\"y_true\": y_true, \"y_pred\": y_pred, \"k\": k}\n\ndef confusion_matrix(y_true, y_pred, k):\n    y_true = np.asarray(y_true)\n    y_pred = np.asarray(y_pred)\n    C = np.zeros((k, k), dtype=int)\n    np.add.at(C, (y_true, y_pred), 1)\n    return C", "confusion_matrix", 20)

---

**Go deeper**
- [Matplotlib: Quick start guide](https://matplotlib.org/stable/users/explain/quick_start.html)
- [Seaborn: tutorial](https://seaborn.pydata.org/tutorial.html)